# Unified Translocation Assay: CZI, OME-Zarr, and OZX

One workflow for local HCS CZI files, OME-Zarr plate directories, and OZX archives. Both existing notebooks remain available. This notebook reuses `translocation_assay_io`, `translocation_plate_layout`, and `translocation_assay_utils`; it does not duplicate reader, segmentation, measurement, or fitting algorithms.

The example measures FKHR-EGFP cytoplasm-to-nucleus translocation in human U2OS cells on a 96-well plate. Wortmannin and LY294002 inhibit PI3 kinase / PKB signaling. DAPI defines nuclei; EGFP nucleus/ring intensity ratios quantify redistribution.

## 1. Configure Environment and Import Libraries

Select the workshop Pixi **default** kernel (`.pixi/envs/default/python.exe` on Windows). Start the notebook from the workshop root or a subdirectory. No packages are installed here. Windows native DLL paths are configured before numerical imports. Restart the kernel and run all cells after changing inputs or shared Python modules.

In [ ]:
import json
import logging
import os
import platform
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from urllib.parse import quote

native_dll_dir = Path(sys.prefix) / "Library" / "bin"
if os.name == "nt" and native_dll_dir.is_dir():
    os.environ["PATH"] = str(native_dll_dir) + os.pathsep + os.environ.get("PATH", "")

current_dir = Path.cwd().resolve()
try:
    workshop_root = next(
        candidate for candidate in (current_dir, *current_dir.parents)
        if (candidate / "pixi.toml").is_file()
        and (candidate / "examples" / "translocation" / "translocation_assay.py").is_file()
    )
except StopIteration as exc:
    raise FileNotFoundError("Start from the workshop root or a subdirectory.") from exc
default_env = workshop_root / ".pixi" / "envs" / "default"
if Path(sys.prefix).resolve() != default_env.resolve():
    raise RuntimeError(f"Select the workshop Pixi default kernel from {default_env}.")
project_dir = workshop_root / "examples" / "translocation"
data_dir = workshop_root / "data" / "translocation"
for module_name in ("translocation_assay_io.py", "translocation_plate_layout.py", "translocation_assay_utils.py"):
    if not (project_dir / module_name).is_file():
        raise FileNotFoundError(f"Missing workshop module: {module_name}")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import itables
import numpy as np
import pandas as pd
import translocation_assay_io as assay_io
import translocation_assay_utils as assay_utils
import translocation_plate_layout as layouts
import xarray as xr
from IPython.display import HTML, IFrame, Image, display
from tqdm.auto import tqdm
from translocation_assay_utils import (
    THRESHOLD_METHODS,
    PlotBackend,
    ThresholdMethod,
    analyze_scenes,
    display_scrollable_table,
    write_analysis_outputs,
)

%matplotlib inline

itables.init_notebook_mode()
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 16)
print(f"Kernel Python: {sys.executable}")
print(f"Workshop: {workshop_root}")
print(f"Data: {data_dir}")

## 2. Select CZI, OME-Zarr, or OZX Input

Uncomment exactly one `input_path` below. An OME-Zarr directory must already exist; the example directory name is illustrative, not created automatically. `input_format = "auto"` detects the container; explicit `"czi"` or `"omezarr"` must match it. OZX is archived OME-Zarr, so use `"omezarr"`, not `"ozx"`, for an explicit override. Remote URLs and single-image NGFF stores are not supported.

`czi_reader_backend` applies only to CZI: `read_stacks` prepares lazy xarray/Dask scenes, while `read_field` reads a well-local field on demand. NGFF always reads full-resolution fields, whether a directory or archive. Inputs remain read-only; OZX is never extracted.

Select exactly one ring-width unit and set the other to `None`. Peak spacing controls watershed seeds, not minimum cell diameter. Set `segmentation_scene` to a mapped field ID or `None`; CZI IDs are stored scene indices, NGFF IDs follow declared well/image order. Default analysis settings match the existing notebooks. Use `seaborn` for embedded PNGs or `plotly` for interactive HTML.

Outputs are separated by input filename, including its extension, under `results/translocation/notebook_unified_run/`. Reruns with the same input overwrite matching exports; change `output_dir` to retain parameter comparisons.

In [ ]:
input_path = data_dir / "transloc_gfp_dapi_0_2micron.czi"
# input_path = data_dir / "transloc_gfp_dapi_0_2micron_ngff_plate_zarr3.ome.zarr"
# input_path = data_dir / "transloc_gfp_dapi_0_2micron_ngff_plate.ozx"
input_format: assay_io.InputFormat = "auto"
czi_reader_backend: assay_io.ReaderBackend = "read_stacks"
output_dir = workshop_root / "results" / "translocation" / "notebook_unified_run" / input_path.name

ring_width_um: float | None = 2.0
ring_width_px: float | None = None
min_nucleus_area_px = 40
min_nucleus_distance_px = 8
threshold_method: ThresholdMethod = "otsu"
ratio_threshold = 2.0
plot_backend: PlotBackend = "seaborn"
segmentation_scene: int | None = 6
channel_indices = {"EGFP": 0, "DAPI": 1}

print(f"Input: {input_path} (requested format: {input_format})")
print(f"Outputs: {output_dir}")
print(f"CZI reader preference: {czi_reader_backend}")
print(f"DAPI threshold method: {threshold_method}")

## 3. Resolve Plate Layout

Select exactly one source: an Excel workbook, or `None` for the manual definitions. Missing or invalid workbooks raise errors; sources are never merged and there is no silent fallback. The resolved read-only table is shown in a scrollable offline ITables preview.

Manual dose lists must match the dose columns. Special conditions apply last, so A1 overrides its negative-control assignment. The manual example explicitly includes blank column-2 assignments for complete acquired-well coverage. Empty control lists or special-condition dictionaries disable their assignments. Treatments come from this layout, not from NGFF image metadata. Rerun this cell and downstream cells after changing assignments.

In [ ]:
plate_layout_path: Path | None = data_dir / "plate_layout_example.xlsx"

resolved_plate_layout_path = plate_layout_path.resolve() if plate_layout_path is not None else None
if resolved_plate_layout_path is not None:
    plate_layout = layouts.load_plate_layout(resolved_plate_layout_path)
    plate_layout_source = str(resolved_plate_layout_path)
else:
    DOSE_WO = [0.977, 1.95, 3.91, 7.81, 15.63, 31.25, 62.5, 125.0, 250.0]
    DOSE_LY = [0.31, 0.63, 1.25, 2.5, 5.0, 10.0, 20.0, 40.0, 80.0]
    WELLS_WO = ["A", "B", "C", "D"]
    WELLS_LY = ["E", "F", "G", "H"]
    DOSE_COLUMNS = ["3", "4", "5", "6", "7", "8", "9", "10", "11"]
    DOSE_SERIES = (
        ("Wortmannin", WELLS_WO, DOSE_WO, "nM"),
        ("LY294002", WELLS_LY, DOSE_LY, "uM"),
    )
    POSITIVE_CONTROL_WELLS = [f"{row}12" for row in WELLS_WO] + [f"{row}1" for row in WELLS_LY]
    NEGATIVE_CONTROL_WELLS = [f"{row}1" for row in WELLS_WO] + [f"{row}12" for row in WELLS_LY]
    SPECIAL_CONDITIONS = {
        **{f"{row}2": "Blank or unassigned" for row in "ABCDEFGH"},
        "A1": "Special condition (39688; unit unspecified)",
    }
    plate_layout = layouts.build_plate_layout(
        dose_series=DOSE_SERIES,
        dose_columns=DOSE_COLUMNS,
        positive_control_wells=POSITIVE_CONTROL_WELLS,
        negative_control_wells=NEGATIVE_CONTROL_WELLS,
        special_conditions=SPECIAL_CONDITIONS,
    )
    plate_layout_source = "Manual notebook configuration"

layout_table = pd.DataFrame([{**assignment, "Well": well} for well, assignment in plate_layout.items()])
print(f"Plate layout: {plate_layout_source} ({len(plate_layout)} wells)")
itables.show(
    layout_table, showIndex=False, paging=False, ordering=False, connected=False,
    layout={"topStart": None, "topEnd": None, "bottomStart": None, "bottomEnd": None},
    scrollY="420px", scrollX=True,
)

## 4. Validate Settings and Load Metadata

The shared adapter detects the container and loads metadata. Only CZI runs `czitools.hcs_check`; its report is shown in a scrollable panel. OME-Zarr directories and OZX archives instead validate declared HCS wells, unique field paths, axes, channel labels/colors, and consistent physical XY spacing.

NGFF supports flat axes/transforms and nested NGFF 0.6 coordinate systems. Non-singleton extra dimensions are rejected, not projected. Missing metadata are errors; channels and scaling are never guessed. Metadata preparation does not read all field pixels. Sources are opened read-only.

In [ ]:
from contextlib import redirect_stdout
from html import escape
from io import StringIO

detected_input_format = assay_io.resolve_input_format(input_path, input_format)
if (ring_width_um is None) == (ring_width_px is None):
    raise ValueError("Specify exactly one ring width in micrometers or pixels.")
requested_ring_width = ring_width_um if ring_width_um is not None else ring_width_px
if requested_ring_width is None or not np.isfinite(requested_ring_width) or requested_ring_width <= 0:
    raise ValueError("Ring width must be finite and positive.")
for setting in (min_nucleus_area_px, min_nucleus_distance_px):
    if isinstance(setting, bool) or not isinstance(setting, int) or setting < 1:
        raise ValueError("Nucleus area and peak spacing must be positive integers.")
if threshold_method not in THRESHOLD_METHODS:
    raise ValueError(f"Unsupported threshold method: {threshold_method}")
if not np.isfinite(ratio_threshold) or ratio_threshold <= 0:
    raise ValueError("The translocation ratio cutoff must be finite and positive.")
if plot_backend not in ("seaborn", "plotly"):
    raise ValueError("Choose the seaborn or plotly backend.")
if czi_reader_backend not in ("read_stacks", "read_field"):
    raise ValueError("Choose read_stacks or read_field for the CZI preference.")
if segmentation_scene is not None and (
    isinstance(segmentation_scene, bool) or not isinstance(segmentation_scene, int) or segmentation_scene < 0
):
    raise ValueError("Preview field ID must be a nonnegative integer or None.")

if detected_input_format == "czi":
    hcs_report = StringIO()
    try:
        with redirect_stdout(hcs_report):
            assay_io.run_hcs_check(input_path)
    finally:
        display(HTML(
            '<div style="max-height: 360px; max-width: 100%; overflow: auto;">'
            f'<pre style="margin: 0; white-space: pre;">{escape(hcs_report.getvalue())}</pre>'
            '</div>'
        ))
assay_input = assay_io.load_assay_input(input_path, input_format, czi_reader_backend)
metadata = assay_input.metadata
reader_backend = assay_input.reader_backend
if metadata.hcs is None:
    raise ValueError("The input must contain usable HCS plate metadata.")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Detected format: {assay_input.input_format}; reader: {reader_backend}")
print(f"HCS wells: {len(metadata.hcs.wells)}")

## 5. Map Fields to Wells

All acquired wells must be covered by the resolved layout. Multiple fields from one well remain separate during processing and are pooled for well summaries.

For CZI, `SceneIndex` preserves the stored zero-based scene ID and `FieldPath` describes its HCS well/local field selector. For NGFF, IDs follow declared plate-well / well-image order; `FieldPath` is the exact image-group path. These NGFF IDs must not be interpreted as original CZI scene IDs. Original well paths such as `A/01` remain available in summaries; canonical names such as `A1` identify treatments. The mapping is exported as `field_mapping.csv`.

In [ ]:
scene_wells = assay_input.scene_wells
unassigned_wells = sorted(set(scene_wells.values()) - set(plate_layout))
if unassigned_wells:
    raise ValueError(f"Acquired wells are missing from the layout: {unassigned_wells}")
scene_table = pd.DataFrame([
    {"SceneIndex": index, "Well": well, "FieldPath": assay_input.field_paths[index]}
    for index, well in sorted(scene_wells.items())
])
print(f"Plate layout: {plate_layout_source}")
print(f"Mapped fields: {len(scene_wells)}")
print(f"Field identifiers: {assay_input.field_index_meaning}")
display_scrollable_table(scene_table)
print("All acquired wells have treatment assignments.")
if segmentation_scene is not None and segmentation_scene not in scene_wells:
    raise ValueError(f"Preview field ID {segmentation_scene} is not mapped to an HCS well.")

## 6. Verify Scaling and Channels

The adapter provides validated **(Y, X)** pixel spacing in micrometers from CZI metadata or full-resolution NGFF dataset transforms and axis units. Physical-width rings account for anisotropic pixels; pixel-width rings use fixed image-pixel distances. Both equivalents are displayed and recorded. At 0.2 micrometers per pixel, a 2-micrometer ring spans 10 pixels along each axis.

EGFP is measured and DAPI defines nuclei. Check the configured indices against stored channel labels, ignoring surrounding whitespace and case. Stored display colors are retained for the exported microfilm composite. Missing channel identities or scaling are not guessed.

In [ ]:
pixel_spacing_um = assay_input.pixel_spacing_um
if not all(np.isfinite(value) and value > 0 for value in pixel_spacing_um):
    raise ValueError(f"Invalid XY pixel spacing: {pixel_spacing_um}")
if ring_width_um is not None:
    ring_width_equivalent_um_yx = (ring_width_um, ring_width_um)
    ring_width_equivalent_px_yx = tuple(ring_width_um / spacing for spacing in pixel_spacing_um)
    ring_width_label = (
        f"{ring_width_um:g} um "
        f"({ring_width_equivalent_px_yx[0]:g} px Y, {ring_width_equivalent_px_yx[1]:g} px X)"
    )
else:
    if ring_width_px is None:
        raise ValueError("Select one ring width before computing equivalents.")
    ring_width_equivalent_px_yx = (ring_width_px, ring_width_px)
    ring_width_equivalent_um_yx = tuple(ring_width_px * spacing for spacing in pixel_spacing_um)
    ring_width_label = (
        f"{ring_width_px:g} px "
        f"({ring_width_equivalent_um_yx[0]:g} um Y, {ring_width_equivalent_um_yx[1]:g} um X)"
    )
display_scrollable_table(pd.DataFrame({
    "Axis": ["Y", "X"],
    "SpacingUmPerPixel": pixel_spacing_um,
    "RingEquivalentUm": ring_width_equivalent_um_yx,
    "RingEquivalentPx": ring_width_equivalent_px_yx,
}))
print(f"Ring label: {ring_width_label}")

if metadata.channelinfo is None:
    raise ValueError("Channel metadata are unavailable.")
channel_names = list(metadata.channelinfo.names)
for channel, index in channel_indices.items():
    if isinstance(index, bool) or not isinstance(index, int) or not 0 <= index < len(channel_names):
        raise ValueError(f"Invalid index {index} for {channel}: {channel_names}")
    if channel_names[index].strip().casefold() != channel.casefold():
        raise ValueError(f"Expected {channel} at index {index}; found {channel_names[index]}.")
display_scrollable_table(pd.DataFrame([
    {"Channel": channel, "Index": index, "StoredName": channel_names[index],
     "DisplayColor": metadata.channelinfo.colors[index]}
    for channel, index in channel_indices.items()
]))

## 7. Preview Fields and Tune Segmentation

The shared preview reader loads only the selected source field. Stackview displays EGFP in green and DAPI in blue. Display contrast does not modify measured raw intensities. With `segmentation_scene = None`, this raw preview uses the first field; the exported segmentation preview selects the first field containing nuclei.

The tuner starts in that field's well. Select another acquired well and a one-based field within it; the mapped source ID is shown separately. Threshold methods are **Otsu, Li, Yen, Triangle, Isodata, and Mean**. Each uses finite DAPI pixels to compute one global threshold per field; constant fields produce no nuclei. All methods share small-object removal, hole filling, and watershed splitting.

Minimum nucleus area, watershed peak spacing, ring width, and responder cutoff start from configuration. Peak spacing controls seed separation, not a guaranteed gap between nuclei. Rings exclude nuclei and contacts with other labels; there is no configurable minimum ring-to-ring gap. Sliders update on release, reusing cached EGFP/DAPI planes for the selected field.

White outlines mark nuclei; magenta outlines mark rings. Solid yellow fills nuclei with finite nucleus/ring ratios at or above the cutoff. Overlay toggles are independent. Changing the cutoff changes classification, not segmentation.

**Tuning is exploratory:** changes persist while switching preview fields, but do not modify analysis configuration, results, or exports. Inspect representative fields, explicitly copy chosen values into section 2, then rerun downstream cells.

In [ ]:
example_scene = segmentation_scene if segmentation_scene is not None else min(scene_wells)
example_reader = assay_input.preview_reader
example_field = example_reader(example_scene)
example_channels = {
    channel: np.asarray(example_field.isel(C=channel_indices[channel]).squeeze(drop=True).values)
    for channel in ("EGFP", "DAPI")
}
if any(image.ndim != 2 for image in example_channels.values()):
    raise ValueError("The preview requires 2D EGFP and DAPI planes.")
print(f"Field ID: {example_scene}; well: {scene_wells[example_scene]}")
print(f"Source selector: {assay_input.field_paths[example_scene]}")
example_view = assay_utils.make_channel_preview(
    egfp=example_channels["EGFP"], dapi=example_channels["DAPI"],
)
display(example_view)

field_tuner = assay_utils.make_field_tuner(
    scene_wells[example_scene],
    field_reader=example_reader,
    scene_wells=scene_wells,
    channel_indices=channel_indices,
    pixel_spacing_um=pixel_spacing_um,
    min_nucleus_area_px=min_nucleus_area_px,
    min_nucleus_distance_px=min_nucleus_distance_px,
    ring_width_um=ring_width_um,
    ring_width_px=ring_width_px,
    ratio_threshold=ratio_threshold,
    threshold_method=threshold_method,
)
display(field_tuner)

## 8. Prepare Reader and Analyze Fields

`prepare_scene_inputs()` returns lazy CZI scene arrays or an on-demand field callback. NGFF reopens the directory/archive read-only for each full-resolution field and closes handles afterward; no plate-wide image array or extraction is needed. Progress reports completed fields rather than source IDs.

The unchanged `analyze_scenes()` pipeline:

1. Loads one field and extracts matching 2D EGFP/DAPI planes.
2. Segments DAPI using the selected global threshold, small-object filtering, hole filling, distance-transform peaks, and watershed.
3. Assigns nearby cytoplasmic pixels to their nearest nucleus within the configured distance, excluding nuclei and contacts with different labels.
4. Measures raw mean EGFP in each nucleus and ring. Ratios are invalid for empty rings or nonpositive ring means.
5. Pools fields by well and finite ratios by drug/dose. Ratios **at or above** the cutoff count as responders.
6. Fits an increasing bounded four-parameter Hill curve to pooled valid-cell fractions, not replicate-weighted well averages.

Fits require at least four finite positive doses and variable responses. No background subtraction, control normalization, or confidence intervals are added. Rings are local cytoplasmic proxies, not whole-cell masks, and can be truncated by crowding or field edges. Memory includes a loaded field, masks, accumulated measurements, and preview data.

In [ ]:
preparation_label = "Prepare lazy scenes" if reader_backend == "read_stacks" else "Prepare field reader"
with tqdm(total=None, desc=preparation_label, unit="field") as preparation_progress:
    scene_arrays, dimensions, scene_count, field_reader = assay_input.prepare_scene_inputs()
    if scene_arrays is not None and not all(isinstance(scene, xr.DataArray) for scene in scene_arrays):
        raise TypeError("Expected xarray scene arrays.")
    preparation_progress.reset(total=scene_count)
    preparation_progress.update(scene_count)
print(f"Prepared {scene_count} fields with {reader_backend}; axes: {dimensions}")
if scene_arrays is not None:
    print(f"First scene sizes: {dict(scene_arrays[0].sizes)}")
else:
    print("Analysis will load one field at a time from the source mapping.")

with tqdm(total=scene_count, desc="Process fields", unit="field") as processing_progress:
    def update_processing_progress(completed: int, total: int) -> None:
        """Update tqdm using cumulative field counts, not source scene IDs."""
        processing_progress.total = total
        processing_progress.update(completed - processing_progress.n)

    results = analyze_scenes(
        metadata=metadata,
        scene_arrays=scene_arrays,
        scene_wells=scene_wells,
        channel_indices=channel_indices,
        pixel_spacing_um=pixel_spacing_um,
        dimensions=dimensions,
        min_nucleus_area_px=min_nucleus_area_px,
        min_nucleus_distance_px=min_nucleus_distance_px,
        threshold_method=threshold_method,
        ratio_threshold=ratio_threshold,
        ring_width_um=ring_width_um,
        ring_width_px=ring_width_px,
        plate_layout=plate_layout,
        segmentation_scene=segmentation_scene,
        progress_callback=update_processing_progress,
        field_reader=field_reader,
    )
print(f"Processed {results.fields_processed} fields; detected {len(results.cell_table):,} nuclei.")

## 9. Export Results and Provenance

The shared writer exports cell, well, dose, and EC50 CSV tables; count and ratio heatmaps; and segmentation/dose-response figures when available. `analysis_parameters.json` describes the completed run, not input configuration.

This notebook also exports `field_mapping.csv` and `plate_layout_used.csv` for **either** layout source. `notebook_provenance.json` records requested/detected formats, actual reader, field-index meaning, NGFF level 0 when applicable, analysis settings, layout, mapping, Pixi interpreter, package versions, and UTC time. Missing values in the provenance layout are serialized as JSON null.

Keep the original input, workbook if used, source modules, manifest/lockfile, and exported files together. No full images are embedded in provenance. Both original notebooks remain unchanged.

In [ ]:
scene_table.to_csv(output_dir / "field_mapping.csv", index=False)
layout_table.to_csv(output_dir / "plate_layout_used.csv", index=False)
input_parameters = {
    "InputPath": str(assay_input.path.resolve()),
    "InputFormat": assay_input.input_format,
    "InputFormatRequested": input_format,
    "ReaderBackend": reader_backend,
    "CziReaderBackendRequested": czi_reader_backend,
    "NgffResolutionLevel": 0 if assay_input.input_format == "omezarr" else None,
    "FieldIndexMeaning": assay_input.field_index_meaning,
    "FieldMappingFile": "field_mapping.csv",
}
parameters = {
    **input_parameters,
    "Channels": {channel: channel_names[index] for channel, index in channel_indices.items()},
    "ChannelIndices": channel_indices,
    "PixelSpacingUmYX": list(pixel_spacing_um),
    "RingWidthUm": ring_width_um,
    "RingWidthPx": ring_width_px,
    "RingWidthEquivalentUmYX": list(ring_width_equivalent_um_yx),
    "RingWidthEquivalentPxYX": list(ring_width_equivalent_px_yx),
    "RingWidthLabel": ring_width_label,
    "MinNucleusAreaPx": min_nucleus_area_px,
    "MinNucleusDistancePx": min_nucleus_distance_px,
    "DapiThresholdMethod": threshold_method,
    "TranslocationRatioThreshold": ratio_threshold,
    "PlotBackend": plot_backend,
    "PlateLayoutSource": plate_layout_source,
    "PlateLayoutSnapshot": "plate_layout_used.csv",
    "PlateLayoutFile": str(resolved_plate_layout_path) if resolved_plate_layout_path is not None else None,
    "SegmentationSceneRequested": segmentation_scene,
    "SegmentationSceneShown": results.segmentation_example[4] if results.segmentation_example is not None else None,
    "FieldsProcessed": results.fields_processed,
    "ObjectsDetected": len(results.cell_table),
}
parameters["InputCzi" if assay_input.input_format == "czi" else "InputOmeZarr"] = str(assay_input.path.resolve())
write_analysis_outputs(results, output_dir, parameters, plot_backend)

package_names = [
    "czitools", "pylibCZIrw", "zarr", "numpy", "pandas", "scipy", "scikit-image",
    "xarray", "dask", "matplotlib", "seaborn", "plotly", "microfilm", "rich",
    "ipykernel", "openpyxl", "itables", "tqdm", "stackview", "ipywidgets",
]
provenance = {
    **parameters,
    "Notebook": "translocation_assay_unified.ipynb",
    "ExportedAtUtc": datetime.now(UTC).isoformat(),
    "PixiEnvironment": "default",
    "PythonExecutable": sys.executable,
    "PythonVersion": platform.python_version(),
    "Platform": platform.platform(),
    "Packages": {name: version(name) for name in package_names},
    "FieldMapping": scene_table.to_dict(orient="records"),
    "PlateLayout": json.loads(layout_table.to_json(orient="records")),
}
(output_dir / "notebook_provenance.json").write_text(
    json.dumps(provenance, indent=2, allow_nan=False), encoding="utf-8",
)
print(f"Exported to {output_dir}")
display_scrollable_table(pd.DataFrame([
    {"File": path.name, "Bytes": path.stat().st_size}
    for path in sorted(output_dir.iterdir()) if path.is_file()
]))

## 10. Display Tables and Figures

`object_preview_well` filters object measurements independently of `well_preview_row`, which filters well summaries. Use `None` to remove either filter. Selected plate rows show **all** matching wells, and dose summaries always show **all** rows in scrollable tables. `table_preview_rows` limits object/EC50 previews and unfiltered well previews only. These settings never change analysis or exported CSVs; rerun just the display cells after changing them.

Heatmaps show retained nucleus counts and median valid nucleus/ring ratios. Inspect spatial trends, low-count wells, and multiple segmentation fields before interpreting responses. The microscopy composite uses metadata colors on black, with nucleus/ring boundaries. Dose curves use pooled valid-cell fractions; normalized panels use fitted bottom/top, not controls. EC50 is the half-amplitude dose of the increasing fit. Nuclei in one well are not independent biological replicates.

Seaborn PNGs are embedded and remain visible when executed notebook outputs are saved. Plotly HTML is shown in local IFrames, not embedded: retain exported HTML with `plotly.min.js`, serve notebook and exports under the same Jupyter root, and trust the notebook. Static viewers may not render HTML. Optional figures are checked against **current** results before file existence, preventing old optional exports from being shown.

For reproducibility, restart and run all, check format/backend, field mapping, channels, scaling, layout, and segmentation sensitivity. Use separate directories for comparisons. See the [example README](README.md) for CLI usage and [workshop README](../../README.md) for setup; the CLI also supports CZI, OME-Zarr directories, and OZX.

In [ ]:
object_preview_well: str | None = "A3"
well_preview_row: str | None = "B"
table_preview_rows = 10

if object_preview_well is not None:
    if not isinstance(object_preview_well, str) or not object_preview_well.strip():
        raise ValueError("Object preview well must be a nonempty name or None.")
    object_preview_well = object_preview_well.strip().upper()
if well_preview_row is not None:
    if (
        not isinstance(well_preview_row, str)
        or not well_preview_row.strip().isascii()
        or not well_preview_row.strip().isalpha()
    ):
        raise ValueError("Well preview row must contain plate-row letters or be None.")
    well_preview_row = well_preview_row.strip().upper()
if isinstance(table_preview_rows, bool) or not isinstance(table_preview_rows, int) or table_preview_rows < 1:
    raise ValueError("Table preview rows must be a positive integer.")

object_preview_table = results.cell_table
if object_preview_well is not None and not object_preview_table.empty:
    object_preview_table = object_preview_table.loc[object_preview_table["Well"].eq(object_preview_well)]
print(
    f"Object preview: {object_preview_well if object_preview_well is not None else 'all wells'}; "
    f"{len(object_preview_table):,} matching objects of {len(results.cell_table):,} total."
)
well_preview_table = results.well_table
if well_preview_row is not None and not well_preview_table.empty:
    well_preview_table = well_preview_table.loc[well_preview_table["Row"].eq(well_preview_row)]
print(
    f"Well summary: {well_preview_row if well_preview_row is not None else 'all rows'}; "
    f"{len(well_preview_table):,} matching wells of {len(results.well_table):,} total."
)
for table_name, table, max_rows in (
    ("Object measurements", object_preview_table, table_preview_rows),
    ("Well summaries", well_preview_table, None if well_preview_row is not None else table_preview_rows),
    ("Dose summaries", results.dose_table, None),
    ("EC50 fits", results.ec50_table, table_preview_rows),
):
    display(HTML(f"<h3>{table_name} ({len(table):,} rows)</h3>"))
    if table.empty:
        if table_name == "Object measurements" and object_preview_well is not None:
            print(f"No object measurements for well {object_preview_well}.")
        elif table_name == "Well summaries" and well_preview_row is not None:
            print(f"No well summaries for row {well_preview_row}.")
        else:
            print("No results available for this table.")
    else:
        display_scrollable_table(table, max_rows=max_rows, show_row_count=True)
if not results.cell_table.empty:
    valid_ratio_count = int(np.isfinite(results.cell_table["TranslocationRatio"]).sum())
    print(f"Finite object ratios: {valid_ratio_count:,} / {len(results.cell_table):,}")

In [ ]:
figure_specs = [
    ("heatmap_number_of_objects", "Nuclei per well", not results.well_table.empty),
    ("heatmap_translocation_ratio", "Median nucleus/ring ratio", not results.well_table.empty),
    ("segmentation_example", "Channel composite and segmentation", results.segmentation_example is not None),
    ("dose_response_ec50", "Dose-response curves and EC50", not results.dose_table.empty),
]
figure_extension = ".png" if plot_backend == "seaborn" else ".html"
for stem, title, available in figure_specs:
    display(HTML(f"<h3>{title}</h3>"))
    figure_path = output_dir / f"{stem}{figure_extension}"
    if not available:
        print("No figure available for the current results.")
    elif not figure_path.is_file():
        print(f"Figure was not exported: {figure_path.name}. Check export warnings.")
    elif plot_backend == "seaborn":
        display(Image(filename=str(figure_path), width=1100))
    else:
        relative_path = Path(os.path.relpath(figure_path, project_dir)).as_posix()
        display(IFrame(src=quote(relative_path, safe="/"), width="100%", height=900))
        print(f"Interactive file: {figure_path}")